# 03 — M1 and M2 Modeling

In [ ]:
from pathlib import Path
import pandas as pd
from src.config import load_config
from src.feature_engineering import get_feature_columns
from src.model_m1 import build_m1_model, split_train_test
from src.model_m2 import fit_m2, predict_m2
from src.labels import build_meta_labels

cfg = load_config(Path('..') / 'config' / 'config.yaml')
panel = pd.read_parquet(Path('..') / cfg.paths.features / 'model_panel.parquet')
if not isinstance(panel.index, pd.MultiIndex):
    panel = panel.set_index(['date', 'ticker'])
feature_cols = get_feature_columns(panel)
train, test = split_train_test(panel, cfg)
m1 = build_m1_model(cfg)
m1.fit(train[feature_cols].fillna(0), train['m1_target'])
signals = m1.predict_signal(panel[feature_cols].fillna(0))
scores = m1.predict_score(panel[feature_cols].fillna(0))
panel = build_meta_labels(panel, signals, scores, cfg)
m2, _ = fit_m2(panel, cfg)
panel = predict_m2(m2, panel, cfg)
panel[['M1_signal', 'M1_score', 'p_success']].head()